In [1]:
# 编码器: 把文本编码成数字的工具

In [8]:
sentence = 'hello everyone , today is a good day .'

In [9]:
# 字典, 也叫做词表 vocabulary
vocab = {
    '<SOS>': 0,
    '<EOS>': 1,
    'hello': 2,
    'everyone': 3,
    'today': 4,
    'is': 5,
    'a': 6,
    'good': 7,
    'day': 8,
    ',': 9,
    '.': 10
}

In [10]:
sent = '<SOS> ' + sentence + ' <EOS>'
print(sent)

<SOS> hello everyone , today is a good day . <EOS>


In [11]:
# 英文分词, 比较简单, 直接按照空格区分就可以
# 中文可以使用分词工具, 比如jieba分词
words = sent.split()
print(words)

['<SOS>', 'hello', 'everyone', ',', 'today', 'is', 'a', 'good', 'day', '.', '<EOS>']


In [12]:
[vocab[i] for i in words]

[0, 2, 3, 9, 4, 5, 6, 7, 8, 10, 1]

### 1 使用编码工具

In [14]:
# 模型和它的编码器是成对使用的, 你使用什么模型, 就它提供的编码器.
# 编码器的名字一般和模型的名字是一样的
# bert-base-chinese
from transformers import BertTokenizer

In [15]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [16]:
tokenizer = BertTokenizer.from_pretrained(
    pretrained_model_name_or_path='bert-base-chinese',
    cache_dir = None,
    force_download = False
)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [17]:
sents = [
    '你站在桥上看风景',
    '看风景的人在楼上看你',
    '明月装饰了你的窗子',
    '你装饰了别人的梦'
]

In [21]:
# 基本的编码函数
out = tokenizer.encode(
    text=sents[0],
    text_pair=sents[1],
    # 句子太长就截断到max_length
    truncation=True,
    # 句子不够长就padding到max_length的长度
    padding='max_length',
    add_special_tokens=True,
    max_length=25,
    return_tensors=None
)
print(out)

[101, 872, 4991, 1762, 3441, 677, 4692, 7599, 3250, 102, 4692, 7599, 3250, 4638, 782, 1762, 3517, 677, 4692, 872, 102, 0, 0, 0, 0]


In [22]:
# 把数据还原成字符串
tokenizer.decode(out)

'[CLS] 你 站 在 桥 上 看 风 景 [SEP] 看 风 景 的 人 在 楼 上 看 你 [SEP] [PAD] [PAD] [PAD] [PAD]'

In [29]:
out = tokenizer(
    text=sents[0],
    text_pair=sents[1],
    # 句子太长就截断到max_length
    truncation=True,
    # 句子不够长就padding到max_length的长度
    padding='max_length',
    # 是否添加特殊字符, [cls]
    add_special_tokens=True,
    max_length=25,
    # 返回的数据类型, 默认返回列表, 可以返回Tensorboardflow, pytorch的tensor
    return_tensors=None,
    # 0, 1 表示是哪个句子的数据
    return_token_type_ids=True,
    # 有用的部分标1, pad部分标0
    return_attention_mask=True,
    # 特殊字符标1, 其他位置标0
    return_special_tokens_mask=True,
    # 返回句子长度
    return_length=True
)

In [30]:
for k, v in out.items():
    print(k, ":", v)

input_ids : [101, 872, 4991, 1762, 3441, 677, 4692, 7599, 3250, 102, 4692, 7599, 3250, 4638, 782, 1762, 3517, 677, 4692, 872, 102, 0, 0, 0, 0]
token_type_ids : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]
attention_mask : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]
special_tokens_mask : [1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1]
length : [25]


In [33]:
# 批量编码函数
out = tokenizer(
    # 句子对
    text=[sents[0], sents[1]],
    text_pair=[sents[2], sents[3]],
    truncation=True,
    padding='max_length',
    add_special_tokens=True,
    max_length=25,
    return_tensors=None,
    return_token_type_ids=True,
    return_attention_mask=True,
    return_special_tokens_mask=True,
    return_length=True
)

In [34]:
for k, v in out.items():
    print(k, ":", v)

input_ids : [[101, 872, 4991, 1762, 3441, 677, 4692, 7599, 3250, 102, 3209, 3299, 6163, 7652, 749, 872, 4638, 4970, 2094, 102, 0, 0, 0, 0, 0], [101, 4692, 7599, 3250, 4638, 782, 1762, 3517, 677, 4692, 872, 102, 872, 6163, 7652, 749, 1166, 782, 4638, 3457, 102, 0, 0, 0, 0]]
token_type_ids : [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]]
attention_mask : [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]]
special_tokens_mask : [[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1], [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1]]
length : [25, 25]


In [37]:
# 字典的操作
vocab = tokenizer.get_vocab()

In [38]:
vocab

{'になる': 9322,
 '##86': 9219,
 'maria': 12096,
 '48': 8214,
 '痹': 4591,
 '瀑': 4105,
 '輓': 6736,
 '##両': 13752,
 '##rite': 11366,
 '[unused69]': 69,
 '##达': 19866,
 'み': 568,
 '##錮': 20153,
 '##雨': 20490,
 'amazon': 8954,
 '##mine': 12723,
 '##諮': 19381,
 'control': 8809,
 '##15': 8493,
 '##裡': 19231,
 '##lone': 12519,
 '##瑞': 17505,
 '##ris': 9848,
 'ᵐ': 334,
 '墜': 1871,
 '##gnet': 13308,
 '##爻': 17328,
 'faq1': 13006,
 'nba': 8391,
 '##掷': 16037,
 '##龟': 21048,
 '爱': 4263,
 '殁': 3649,
 '麾': 7940,
 'super': 8988,
 'living': 12113,
 '##沂': 16809,
 '##‖': 13496,
 '##桑': 16490,
 'think': 12553,
 '##陕': 20419,
 '摒': 3034,
 '##鹂': 20954,
 '##哑': 14571,
 '##沮': 16832,
 '##準': 17033,
 '涌': 3869,
 '##工': 15396,
 '52sykb': 12846,
 '##秸': 17975,
 '蹟': 6695,
 '8cm': 12237,
 '饲': 7654,
 '墻': 1878,
 '谓': 6458,
 '##藻': 19033,
 '##坠': 14842,
 '##鄧': 20028,
 '淼': 3925,
 '##阔': 20390,
 '##讓': 19423,
 'why': 11177,
 '##価': 13957,
 '颍': 7572,
 '胳': 5538,
 '##裸': 19237,
 '##咋': 14525,
 '迴': 6836,
 'в': 235

In [39]:
len(vocab)

21128

In [40]:
# bert_base_chinese 是把每个中文当成一个词
'明月' in vocab

False

In [41]:
# 添加新词
tokenizer.add_tokens(new_tokens=['明月', '装饰', '窗子'])

3

In [42]:
# 添加特殊字符
tokenizer.add_special_tokens({'eos_token':'[EOS]'})

1

In [44]:
for word in ['明月', '装饰', '窗子','[EOS]']:
    print(tokenizer.get_vocab()[word])

21128
21129
21130
21131


In [45]:
# 用新词表去编码
out = tokenizer.encode(text='明月装饰了你的窗子[EOS]',
                text_pair=None,
                 truncation=True,
                 padding='max_length',
                 add_special_tokens=True,
                 max_length=10,
                 return_tensors=None
                )
print(out)

[101, 21128, 21129, 749, 872, 4638, 21130, 21131, 102, 0]


In [46]:
tokenizer.decode(out)

'[CLS] 明月 装饰 了 你 的 窗子 [EOS] [SEP] [PAD]'

In [47]:
# 总结 编码器工作流程: 定义字典, 句子预处理, 分词, 编码